In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO
import matplotlib.colors as mcolors

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd())))

In [ ]:
def plot_algo_rank(algo_rank, colors):
    """
    Plots the Algorithm Ranking Based on Best SMAPE Counts.
    
    Args:
        algo_rank (pd.): Ranked algorithms with counts of best performances.
        colors (list): List of color hex codes.
        
    Returns:
        None
    """
    plt.figure(figsize=(12, 8))
    bars = plt.bar(algo_rank.index, algo_rank.values, color=colors[:len(algo_rank)])
    
    # Add value labels on top of each bar
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height, f'{height}', 
                 ha='center', va='bottom', fontsize=14)
    
    plt.title('Algorithm Ranking Based on Best SMAPE Counts', fontsize=24, color=colors[1])
    plt.xlabel('Algorithms', fontsize=20, color=colors[1])
    plt.ylabel('Number of Best SMAPE Counts', fontsize=20, color=colors[1])
    plt.xticks(rotation=45, fontsize=14, color=colors[1])
    plt.yticks(fontsize=14, color=colors[1])
    plt.tight_layout()
    plt.grid(False)
    plt.show()
    
def plot_average_smape(average_smape, colors):
    """
    Plots the Average SMAPE per Algorithm.
    
    Args:
        average_smape (pd.): Average SMAPE per algorithm.
        colors (list): List of color hex codes.
        
    Returns:
        None
    """
    print(average_smape)
    plt.figure(figsize=(12, 8))
    bars = plt.bar(average_smape.index, average_smape.values, color=colors[:len(average_smape)])
    
    # Add value labels on top of each bar
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height, f'{height / 100:.2%}', 
                 ha='center', va='bottom', fontsize=14)
    
    plt.title('Average SMAPE per Algorithm', fontsize=24, color=colors[1])
    plt.xlabel('Algorithms', fontsize=20, color=colors[1])
    plt.ylabel('Average SMAPE', fontsize=20, color=colors[1])
    plt.xticks(rotation=45, fontsize=14, color=colors[1])
    plt.yticks(fontsize=14, color=colors[1])
    plt.tight_layout()
    plt.grid(False)
    plt.show()

In [ ]:
def plot_top3_smape_per_well(df, colors):
    """
    Plota os 3 melhores algoritmos por Well, destacando a performance com cores e transparências.
    
    Args:
        df (pd.DataFrame): DataFrame contendo as colunas 'Well', 'Method' e 'SMAPE'.
        colors (list): Lista de códigos hexadecimais para as cores.
        
    Returns:
        None: Exibe o gráfico gerado.
    """
    # Obter os 3 melhores Methods por Well
    top3_per_well = df.groupby('Well').apply(
        lambda x: x.nsmallest(3, 'SMAPE').assign(Rank=range(1, len(x.nsmallest(3, 'SMAPE'))+1))
    ).reset_index(drop=True)
    
    # Definir as cores base para cada Well
    base_colors = ['B22222', '2E2E2E', '206A92', '1E5631', 'E3C800']
    wells = df['Well'].unique()
    well_color_map = {well: base_colors[i % len(base_colors)] for i, well in enumerate(wells)}
    
    # Mapear transparências por ranking
    rank_opacity_map = {1: 1.0, 2: 0.9, 3: 0.8}
    
    # Adicionar colunas de cor e opacidade
    top3_per_well['BaseColor'] = top3_per_well['Well'].map(well_color_map)
    top3_per_well['Opacity'] = top3_per_well['Rank'].map(rank_opacity_map)
    
    # Converter cores hexadecimais para rgba
    def hex_to_rgba(hex_color, opacity):
        hex_color = hex_color.lstrip('#')
        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)
        return f'rgba({r},{g},{b},{opacity})'
    
    top3_per_well['ColorRGBA'] = top3_per_well.apply(
        lambda x: hex_to_rgba(x['BaseColor'], x['Opacity']), axis=1
    )
    
    # Separar dados por ranking
    top1 = top3_per_well[top3_per_well['Rank'] == 1]
    top2 = top3_per_well[top3_per_well['Rank'] == 2]
    top3 = top3_per_well[top3_per_well['Rank'] == 3]
    
    # Criar o gráfico
    fig = go.Figure()
    
    # Adicionar barras para cada ranking com texto rotacionado e fonte maior
    fig.add_trace(
        go.Bar(
            x=top1['Well'],
            y=top1['SMAPE'],
            name='Top 1',
            text=top1['Method'],
            textposition='outside',
            textangle=-90,  # Rotaciona o texto em 90 graus
            textfont=dict(size=20),  # Aumenta o tamanho da fonte
            marker_color=top1['ColorRGBA'],
            offsetgroup='1',
            customdata=top1['Method'],
            hovertemplate='<b>%{customdata}</b><br>SMAPE: %{y:}<extra></extra>'
        )
    )
    
    fig.add_trace(
        go.Bar(
            x=top2['Well'],
            y=top2['SMAPE'],
            name='Top 2',
            text=top2['Method'],
            textposition='outside',
            textangle=-90,
            textfont=dict(size=16),
            marker_color=top2['ColorRGBA'],
            offsetgroup='2',
            customdata=top2['Method'],
            hovertemplate='<b>%{customdata}</b><br>SMAPE: %{y:}<extra></extra>'
        )
    )
    
    fig.add_trace(
        go.Bar(
            x=top3['Well'],
            y=top3['SMAPE'],
            name='Top 3',
            text=top3['Method'],
            textposition='outside',
            textangle=-90,
            textfont=dict(size=14),
            marker_color=top3['ColorRGBA'],
            offsetgroup='3',
            customdata=top3['Method'],
            hovertemplate='<b>%{customdata}</b><br>SMAPE: %{y:}<extra></extra>'
        )
    )
    
    # Atualizar o layout do gráfico
    fig.update_layout(
        title='Top 3 Algoritmos por Well Baseado no SMAPE',
        xaxis_title='Well',
        yaxis_title='SMAPE',
        barmode='group',
        plot_bgcolor='white',
        paper_bgcolor='white',
        autosize=False,
        width=1100,
        height=600,
        legend=dict(
            title='Ranking',
            font=dict(size=18),
            bgcolor='rgba(255,255,255,0)'
        ),
        title_font=dict(size=30),
        xaxis=dict(
            title_font=dict(size=28),
            tickfont=dict(size=20),
            type='category'
        ),
        yaxis=dict(
            title_font=dict(size=28),
            tickfont=dict(size=14),
        ),
        font=dict(
            size=18
        ),
        margin=dict(l=40, r=40, t=60, b=120)
    )
    
    # Mostrar o gráfico
    fig.show()

In [ ]:
def plot_top3_smape_per_well_horizontal(df, colors):
    """
    Plots the top 3 algorithms per well in a horizontal layout, highlighting performance with an innovative design and logarithmic scale.

    Args:
        df (pd.DataFrame): DataFrame containing the columns 'Well', 'Method', and 'SMAPE'.
        colors (list): List of hex color codes.

    Returns:
        None: Displays the generated chart.
    """
    # Get the top 3 methods per well
    top3_per_well = df.groupby('Well').apply(
        lambda x: x.nsmallest(3, 'SMAPE').assign(Rank=range(1, len(x.nsmallest(3, 'SMAPE')) + 1))
    ).reset_index(drop=True)

    # Define base colors for each well
    base_colors = colors

    # Create a sequential mapping of colors based on the sorted unique wells in top3_per_well
    ordered_wells = top3_per_well['Well'].drop_duplicates().reset_index(drop=True)
    well_color_map = {well: base_colors[i % len(base_colors)] for i, well in enumerate(ordered_wells)}


    # Map transparency by rank
    rank_opacity_map = {1: 1.0, 2: 0.9, 3: 0.8}

    # Add color and opacity columns
    top3_per_well['BaseColor'] = top3_per_well['Well'].map(well_color_map)
    top3_per_well['Opacity'] = top3_per_well['Rank'].map(rank_opacity_map)

    # Convert hex colors to rgba
    def hex_to_rgba(hex_color, opacity):
        hex_color = hex_color.lstrip('#')
        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)
        return f'rgba({r},{g},{b},{opacity})'

    top3_per_well['ColorRGBA'] = top3_per_well.apply(
        lambda x: hex_to_rgba(x['BaseColor'], x['Opacity']), axis=1
    )

    # Create the figure
    fig = go.Figure()

    # Add horizontal bars for each rank
    for rank in range(1, 4):
        rank_data = top3_per_well[top3_per_well['Rank'] == rank]
        fig.add_trace(
            go.Bar(
                y=rank_data['Well'],
                x=rank_data['SMAPE'],
                orientation='h',
                name=f'Top {rank}',
                text=rank_data['Method'],
                textposition='inside',
                textfont=dict(size=25, color='white'),
                marker=dict(
                    color=base_colors[rank-1],
                    line=dict(width=0, color='black')
                ),
                hovertemplate='<b>Well: %{y}</b><br>Method: %{text}<br>SMAPE: %{x}<extra></extra>'
            )
        )

    # Add scatter trace to highlight the best SMAPE values
    fig.add_trace(
        go.Scatter(
            y=top3_per_well[top3_per_well['Rank'] == 1]['Well'],
            x=top3_per_well[top3_per_well['Rank'] == 1]['SMAPE'],
            mode='markers+text',
            marker=dict(size=14, color='gold', symbol='star'),
            textposition='top right',
            name='Best Method',
            hovertemplate='<b>Best Well: %{y}</b><br>Method: %{text}<br>SMAPE: %{x}<extra></extra>'
        )
    )

    # Update the layout with a logarithmic scale on the X-axis
    fig.update_layout(
        title=dict(
            text='Top 3 Algorithms per Well Based on SMAPE (Logarithmic Scale)',
            x=0.5,  # Center the title
            font=dict(size=40)
        ),
        xaxis_title='SMAPE (Logarithmic Scale)',
        yaxis_title='Well',
        barmode='stack',
        plot_bgcolor='white',
        paper_bgcolor='white',
        autosize=False,
        width=1500,
        height=900,
        xaxis=dict(
            type='log',  # Activate the logarithmic scale
            title_font=dict(size=30),
            tickfont=dict(size=24),
        ),
        yaxis=dict(
            title_font=dict(size=30),
            tickfont=dict(size=24),
            autorange='reversed',
            categoryorder='total ascending'
        ),
        font=dict(
            size=18
        ),
        margin=dict(l=60, r=60, t=80, b=60),
        showlegend=True,
        legend=dict(
            font=dict(size=16),
            bgcolor='rgba(255,255,255,0.5)'
        )
    )
    
    fig.write_image("Top_3.pdf", width=1500, height=900)

    # Display the chart
    fig.show()

In [ ]:
def plot_top3_smape_per_well_spider(df, colors):
    """
    Plota os 3 melhores algoritmos por Well usando gráficos de radar individuais em subplots,
    com cada gráfico usando sua própria faixa de valores para melhor legibilidade.

    Args:
        df (pd.DataFrame): DataFrame contendo as colunas 'Well', 'Method' e 'SMAPE'.
        colors (list): Lista de códigos hexadecimais para as cores.
            
    Returns:
        None: Exibe o gráfico gerado.
    """
    # Obter os 3 melhores Methods por Well
    top3_per_well = df.groupby('Well').apply(
        lambda x: x.nsmallest(3, 'SMAPE').assign(Rank=range(1, len(x.nsmallest(3, 'SMAPE'))+1))
    ).reset_index(drop=True)
    
    # Lista de Wells únicos
    wells = df['Well'].unique()
    n_wells = len(wells)
    
    # Definir o layout da grade para os subplots
    n_cols = 2  # Número de colunas desejado
    n_rows = math.ceil(n_wells / n_cols)
    
    # Criar subplots com gráficos polares
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        specs=[[{'type': 'polar'} for _ in range(n_cols)] for _ in range(n_rows)],
        subplot_titles=[f'Well {well}' for well in wells]
    )
    
    # Mapear cores para cada Well
    well_color_map = {well: colors[i % len(colors)] for i, well in enumerate(wells)}
    
    # Iterar sobre cada Well e adicionar um gráfico de radar no subplot correspondente
    for i, well in enumerate(wells):
        row = (i // n_cols) + 1
        col = (i % n_cols) + 1
        
        data = top3_per_well[top3_per_well['Well'] == well]
        categories = data['Method'].tolist()
        values = data['SMAPE'].tolist()
        
        # Fechar o gráfico de radar
        categories += [categories[0]]
        values += [values[0]]
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=categories,
            fill='toself',
            name=f'Well {well}',
            line_color=well_color_map[well],
            opacity=0.8,
            showlegend=False
        ), row=row, col=col)
        
        # Determinar o valor máximo do eixo radial para este Well
        max_value = max(values) * 1.1  # Adiciona 10% de folga
        min_value = 0  # Inicia o eixo radial em zero
        
        # Atualizar o layout polar de cada subplot individualmente
        fig.update_polars(
            radialaxis=dict(
                visible=True,
                title='SMAPE',
                range=[min_value, max_value],
                tickfont=dict(size=10),
                titlefont=dict(size=12)
            ),
            angularaxis=dict(
                tickfont=dict(size=10)
            ),
            row=row, col=col
        )
    
    # Ajustar o layout geral
    fig.update_layout(
        title='Comparação dos Top 3 Algoritmos por Well',
        font=dict(size=12),
        title_font=dict(size=16),
        paper_bgcolor='white',
        plot_bgcolor='white',
        height=400 * n_rows,  # Ajusta a altura da figura com base no número de linhas
        margin=dict(l=50, r=50, t=80, b=50)
    )
    
    # Mostrar o gráfico
    fig.show()

In [ ]:
def generate_plots(algo_rank, average_smape, df, colors):
    """
    Generates all required plots.
    
    Args:
        algo_rank (pd.): Ranked algorithms with counts of best performances.
        average_smape (pd.): Average SMAPE per algorithm.
        df (pd.DataFrame): DataFrame containing the data.
        colors (list): List of color hex codes.
        
    Returns:
        None
    """
    plot_algo_rank(algo_rank, colors)
    plot_average_smape(average_smape, colors)
    plot_top3_smape_per_well(df, colors)
    plot_top3_smape_per_well_horizontal(df, colors)
    # plot_top3_smape_per_well_spider(df, colors)

In [ ]:
import pandas as pd
import re
from io import StringIO

def preprocess_input(data_str: str) -> pd.DataFrame:
    """
    Pré-processa o texto que contém dados experimentais em DataFrame padronizado:
    ['Well', 'Method', 'Tipo', 'R²', 'SMAPE', 'MAE'].
    """
    lines = data_str.strip().splitlines()
    data_lines = lines[1:]  # pula o cabeçalho original
    rows = []
    for line in data_lines:
        parts = re.split(r'\s+', line.strip())
        if len(parts) == 7:
            _, Well, Method, Tipo, R2, SMAPE, MAE = parts
        elif len(parts) == 5:
            _, Well, Method, SMAPE, MAE = parts
            Tipo = pd.NA
            R2 = pd.NA
        elif len(parts) == 6:
            # índice, Well, Method, R², SMAPE, MAE
            _, Well, Method, R2, SMAPE, MAE = parts
            Tipo = pd.NA
        else:
            raise ValueError(f"Linha com número inesperado de campos ({len(parts)}): {line}")
        rows.append([Well, Method, Tipo, R2, SMAPE, MAE])
    df = pd.DataFrame(rows, columns=['Well', 'Method', 'Tipo', 'R²', 'SMAPE', 'MAE'])

    # Preenche Well para frente e limpa campos numéricos
    df['Well'] = df['Well'].replace('', pd.NA).ffill()
    df['SMAPE'] = pd.to_numeric(df['SMAPE'].astype(str).str.replace('%', '').str.strip(), errors='coerce')
    df['MAE'] = pd.to_numeric(df['MAE'].astype(str).str.replace(',', '').str.strip(), errors='coerce')
    return df

import pandas as pd

def generate_latex_tables(data_to_create_Table: str) -> tuple[str, str]:
    """
    Gera duas tabelas LaTeX:
      - Uma tabela com MAE e SMAPE (%)
      - Outra tabela apenas com SMAPE (%)
    Inclui linha média de SMAPE e linha média de NMAE (%), que representa o MAE normalizado por poço.
    """
    methods = ['ARIMA', 'AutoARIMA', 'LinearRegression', 'TiDE', 'N-Beats',
               'NHiTS', 'TiDE+RIN', 'NLinear', 'XGB+Ours', 'Ours']

    df = preprocess_input(data_to_create_Table)

    # Garante que todos os métodos existam
    for method in methods:
        if method not in df['Method'].unique():
            df = df.append(
                {'Well': df['Well'].iloc[-1], 'Method': method,
                 'Tipo': pd.NA, 'R²': pd.NA, 'SMAPE': pd.NA, 'MAE': pd.NA},
                ignore_index=True
            )

    # Pivotar o DataFrame
    pivot_table = df.pivot_table(
        index='Well',
        columns='Method',
        values=['MAE', 'SMAPE'],
        aggfunc='first'
    )
    for method in methods:
        if method not in pivot_table.columns.levels[1]:
            pivot_table[('MAE', method)] = pd.NA
            pivot_table[('SMAPE', method)] = pd.NA

    pivot_table = pivot_table.reindex(
        columns=pd.MultiIndex.from_product([['MAE', 'SMAPE'], methods])
    )

    # Melhor SMAPE por poço
    best_smape = pivot_table['SMAPE'].idxmin(axis=1)
    num_methods = len(methods)

    # Calcular NMAE (%): normalizando MAE pelo MAE médio do próprio poço
    mean_mae_per_well = pivot_table['MAE'].mean(axis=1)
    normalized_mae = pivot_table['MAE'].div(mean_mae_per_well, axis=0) * 100
    avg_nmae = normalized_mae.mean(axis=0)

    avg_smape = pivot_table['SMAPE'].mean(axis=0)
    avg_mae = pivot_table['MAE'].mean()

    # ---- Gerar tabela MAE + SMAPE ----
    table_mae_smape = (
        "\\begin{table}[h!]\n"
        "\\centering\n"
        "\\caption{MAE and SMAPE (\\%) comparison at the 56-day horizon.}\n"
        "\\label{tab:smape_mae_56}\n"
        "\\begin{threeparttable}\n"
        "\\resizebox{\\textwidth}{!}{%\n"
        f"\\begin{{tabular}}{{@{{}}l{' cc'*num_methods}@{{}}}}\n"
        "\\toprule\n"
        "\\multirow{2}{*}{\\textbf{Well}} \n"
    )
    for m in methods:
        table_mae_smape += f"& \\multicolumn{{2}}{{c}}{{\\textbf{{{m}}}}}\n"
    table_mae_smape += "\\\\\n"
    cmidrules = " ".join(f"{2*i+2}-{2*i+3}" for i in range(num_methods))
    table_mae_smape += f"\\cmidrule(lr){{{cmidrules}}}\n"
    table_mae_smape += "& " + " & ".join(["\\textbf{MAE} & \\textbf{SMAPE}"]*num_methods) + " \\\\\n"
    table_mae_smape += "\\midrule\n"

    for well in pivot_table.index:
        row = f"\\textbf{{{well}}}"
        for method in methods:
            mae = pivot_table.at[well, ('MAE', method)]
            smape = pivot_table.at[well, ('SMAPE', method)]

            mae_str = f"{mae:,.2f}" if pd.notna(mae) else "-"
            if pd.notna(smape):
                smape_formatted = f"{smape:.2f}\\%"
                smape_str = f"\\textbf{{{smape_formatted}}}" if method == best_smape.loc[well] else smape_formatted
            else:
                smape_str = "-"
            row += f" & {mae_str} & {smape_str}"
        table_mae_smape += row + " \\\\\n"

    avg_row = "\\textbf{Average SMAPE}"
    for method in methods:
        avg_val = avg_smape[method]
        avg_row += " & - & " + (f"{avg_val:.2f}\\%" if pd.notna(avg_val) else "-")
    avg_row += " \\\\\n"

    avg_row_nmae = "\\textbf{Average NMAE}"
    for method in methods:
        nmae_val = avg_nmae[method]
        avg_row_nmae += " & - & " + (f"{nmae_val:.2f}\\%" if pd.notna(nmae_val) else "-")
    avg_row_nmae += " \\\\\n"

    table_mae_smape += "\\midrule\n" + avg_row + avg_row_nmae
    table_mae_smape += (
        "\\bottomrule\n"
        "\\end{tabular}}\n"
        "\\begin{tablenotes}[flushleft]\n"
        "\\item \\textbf{Note:} The best SMAPE (lowest) per row is shown in bold.\n"
        "\\end{tablenotes}\n"
        "\\end{threeparttable}\n"
        "\\end{table}\n"
    )

    # ---- Gerar tabela apenas SMAPE ----
    table_smape = (
        "\\begin{table}[h!]\n"
        "\\centering\n"
        "\\caption{SMAPE (\\%) comparison at the 56-day horizon.}\n"
        "\\label{tab:smape_only}\n"
        "\\begin{threeparttable}\n"
        "\\resizebox{\\textwidth}{!}{%\n"
        f"\\begin{{tabular}}{{@{{}}l{' c'*num_methods}@{{}}}}\n"
        "\\toprule\n"
        "\\textbf{Well} "
    )
    for m in methods:
        table_smape += f"& \\textbf{{{m}}} "
    table_smape += "\\\\\n\\midrule\n"
    for well in pivot_table.index:
        row = f"\\textbf{{{well}}}"
        for method in methods:
            smape = pivot_table.at[well, ('SMAPE', method)]
            smape_str = f"{smape:.2f}\\%" if pd.notna(smape) else "-"
            if pd.notna(smape) and method == best_smape.loc[well]:
                smape_str = f"\\textbf{{{smape_str}}}"
            row += f" & {smape_str}"
        table_smape += row + " \\\\\n"

    avg_row_smape = "\\textbf{Average SMAPE}"
    for method in methods:
        avg_val = avg_smape[method]
        avg_row_smape += " & " + (f"{avg_val:.2f}\\%" if pd.notna(avg_val) else "-")
    avg_row_smape += " \\\\\n"
    
    avg_row_mae = "\\textbf{Average MAE}"
    for method in methods:
        mae_val = avg_mae[method]
        avg_row_mae += " & " + (f"{mae_val:,.2f}" if pd.notna(mae_val) else "-")
    avg_row_mae += " \\\\\n"
    
    table_smape += "\\midrule\n" + avg_row_smape + avg_row_mae

    table_smape += (
        "\\bottomrule\n"
        "\\end{tabular}}\n"
        "\\begin{tablenotes}[flushleft]\n"
        "\\item \\textbf{Note:} The best SMAPE (lowest) per row is shown in bold.\n"
        "\\end{tablenotes}\n"
        "\\end{threeparttable}\n"
        "\\end{table}\n"
    )

    return table_mae_smape, table_smape

import pandas as pd

def generate_latex_tables(data_to_create_Table: str, best_caption_summary: str = "") -> str:
    """
    Generates a LaTeX table for SMAPE (%) comparison matching the target pattern.
    Highlights the best and second-best SMAPE per well.
    Adds summary lines for Median MAE, Average Ranking, and Average SMAPE (with best value in bold).
    Allows for a custom summary string in the caption.
    """
    methods = ['ARIMA', 'AutoARIMA', 'LinearRegression', 'TiDE', 'N-Beats',
               'NHiTS', 'TiDE+RIN', 'NLinear', 'XGB+Ours', 'Ours']
    df = preprocess_input(data_to_create_Table)  # Assumes this returns correct DataFrame

    # Ensure all methods exist per well
    for method in methods:
        for well in df['Well'].unique():
            if not ((df['Well'] == well) & (df['Method'] == method)).any():
                df = pd.concat(
                    [df, pd.DataFrame([{
                        'Well': well, 'Method': method,
                        'Tipo': pd.NA, 'R²': pd.NA, 'SMAPE': pd.NA, 'MAE': pd.NA
                    }])],
                    ignore_index=True
                )

    # Pivot for SMAPE and MAE
    pivot_table = df.pivot_table(
        index='Well',
        columns='Method',
        values=['MAE', 'SMAPE'],
        aggfunc='first'
    ).reindex(columns=pd.MultiIndex.from_product([['MAE', 'SMAPE'], methods]))

    # Compute summary statistics
    avg_smape = pivot_table['SMAPE'].mean()
    median_mae = pivot_table['MAE'].median()
    ranking = pivot_table['MAE'].rank(axis=1, ascending=True)
    avg_rank = ranking.mean(axis=0)

    # --- Start LaTeX ---
    caption = "SMAPE (\\%) comparison at the 56-day horizon."
    if best_caption_summary:
        caption += f" {best_caption_summary}"
    table_latex = (
        "\\begin{table}[h!]\n"
        "\\centering\n"
        f"\\caption{{{caption}}}\n"
        "\\label{tab:smape_mae_56}\n"
        "\\begin{threeparttable}\n"
        "\\resizebox{\\textwidth}{!}{%\n"
        f"\\begin{{tabular}}{{@{{}}l{' c'*len(methods)}@{{}}}}\n"
        "\\toprule\n"
        "\\textbf{Well} "
    )
    for m in methods:
        table_latex += f"& \\textbf{{{m}}} "
    table_latex += "\\\\\n\\midrule\n"

    # --- Well rows with SMAPE ---
    for well in pivot_table.index:
        row = f"\\textbf{{{well}}}"
        smape_values = pivot_table.loc[well, 'SMAPE']
        # Get best and second-best (lowest) SMAPE, ignoring NaNs
        smape_sorted = smape_values.dropna().sort_values()
        if len(smape_sorted) == 0:
            best, second = None, None
        else:
            best = smape_sorted.index[0]
            second = smape_sorted.index[1] if len(smape_sorted) > 1 else None

        for method in methods:
            smape = smape_values[method]
            smape_str = "-" if pd.isna(smape) else f"{smape:.2f}\\%"
            if pd.notna(smape):
                if method == best:
                    smape_str = f"\\textbf{{{smape_str}}}~\\circledgold{{1}}"
                elif method == second:
                    smape_str = f"{smape_str}~\\circledsilver{{2}}"
            row += f" & {smape_str}"
        table_latex += row + " \\\\\n"

    # --- Summary rows ---
    # Helper to find best per summary row (for bolding)
    def best_idx(vals, bold_min=True):
        notna = vals.dropna()
        if len(notna) == 0:
            return []
        if bold_min:
            minval = notna.min()
            return [idx for idx, val in notna.items() if val == minval]
        else:
            maxval = notna.max()
            return [idx for idx, val in notna.items() if val == maxval]

    # Median MAE row
    median_mae_idx = best_idx(median_mae, bold_min=True)
    median_row = "\\textbf{Median MAE}"
    for method in methods:
        val = median_mae[method]
        val_str = "-" if pd.isna(val) else f"{val:,.2f}"
        if method in median_mae_idx and pd.notna(val):
            val_str = f"\\textbf{{{val_str}}}"
        median_row += f" & {val_str}"
    median_row += " \\\\\n"

    # Average Ranking row (lower is better)
    avg_rank_idx = best_idx(avg_rank, bold_min=True)
    rank_row = "\\textbf{Average Ranking}"
    for method in methods:
        val = avg_rank[method]
        val_str = "-" if pd.isna(val) else f"{val:.2f}"
        if method in avg_rank_idx and pd.notna(val):
            val_str = f"\\textbf{{{val_str}}}"
        rank_row += f" & {val_str}"
    rank_row += " \\\\\n"

    # Average SMAPE row
    avg_smape_idx = best_idx(avg_smape, bold_min=True)
    avg_smape_row = "\\textbf{Average SMAPE}"
    for method in methods:
        val = avg_smape[method]
        val_str = "-" if pd.isna(val) else f"{val:.2f}\\%"
        if method in avg_smape_idx and pd.notna(val):
            val_str = f"\\textbf{{{val_str}}}"
        avg_smape_row += f" & {val_str}"
    avg_smape_row += " \\\\\n"

    # Add summary rows
    table_latex += "\\midrule\n"
    table_latex += median_row
    table_latex += rank_row
    table_latex += avg_smape_row

    # --- End table ---
    table_latex += (
        "\\bottomrule\n"
        "\\end{tabular}}\n"
        "\\begin{tablenotes}[flushleft]\\scriptsize\\raggedright\n"
        "\\item \\textbf{Note:} The best SMAPE per well is shown in bold with \\circledgold{1}, and the second-best with \\circledsilver{2}. "
        "Median MAE summarizes typical absolute errors across datasets, and Average Ranking reflects each model's relative performance. "
        "Individual MAE results are accompanied by plots and can be accessed through the program.\n"
        "\\end{tablenotes}\n"
        "\\end{threeparttable}\n"
        "\\end{table}\n"
    )
    return table_latex

In [ ]:
import pandas as pd
from io import StringIO

def parse_data(data_str):
    """
    Parses the input string into a pandas DataFrame.
    
    Args:
        data_str (str): Multiline string containing the data.
        
    Returns:
        pd.DataFrame: Parsed DataFrame with appropriate data types.
    """
    # Use StringIO to read the string as a file
    data = StringIO(data_str)
    
    # Read the data, skipping any empty lines and handling delimiters
    df = pd.read_csv(data, sep='\t', skip_blank_lines=True)
    
    # Convert SMAPE from percentage string to float
    df['SMAPE'] = df['SMAPE'].str.rstrip('%').astype(float)
    
    return df

def rank_algorithms(df):
    """
    Ranks algorithms based on the number of times they have the lowest SMAPE per Well.
    
    Args:
        df (pd.DataFrame): DataFrame containing the data.
        
    Returns:
        pd.: Ranked algorithms with counts of best performances.
    """
    # Group by 'Well' and find the algorithm with the lowest SMAPE
    best_algos = df.loc[df.groupby('Well')['SMAPE'].idxmin()]
    
    print(best_algos)
    
    # Count the occurrences of each algorithm being the best
    algo_rank = best_algos['Method'].value_counts().sort_values(ascending=False)
    
    return algo_rank

def calculate_average_smape(df):
    """
    Calculates the average SMAPE for each algorithm across all Wells.
    
    Args:
        df (pd.DataFrame): DataFrame containing the data.
        
    Returns:
        pd.: Average SMAPE per algorithm.
    """
    average_smape = df.groupby('Method')['SMAPE'].mean().sort_values()
    return average_smape

def generate_summary(algo_rank, average_smape):
    """
    Generates a scientific-style summary of the findings.
    
    Args:
        algo_rank (pd.): Ranked algorithms with counts of best performances.
        average_smape (pd.): Average SMAPE per algorithm.
        
    Returns:
        str: Summary text.
    """
    summary = (
        "The evaluation of regression methods across various wells (Wells) revealed significant differences in performance. "
        "Based on the SMAPE metric, the ranking of algorithms indicates that "
    )
    
    # Top 3 algorithms
    top_algos = algo_rank.head(3)
    top_algo_names = ', '.join(top_algos.index.tolist())
    summary += f"{top_algo_names} were the most frequently top-performing algorithms.\n\n"
    
    summary += "Additionally, the average SMAPE across all wells for each algorithm was calculated. The results show that "
    
    # Top 3 best average SMAPE
    best_avg_algos = average_smape.head(3)
    best_avg_names = ', '.join(best_avg_algos.index.tolist())
    summary += f"{best_avg_names} achieved the lowest average SMAPE values, indicating superior overall accuracy.\n\n"
    
    summary += "These findings suggest that selecting the appropriate regression method is crucial for optimizing prediction accuracy in well performance analysis."
    
    return summary

def main(data_str):
    """
    Main function to process the data and generate the summary.
    
    Args:
        data_str (str): Multiline string containing the data.
        
    Returns:
        None
    """
    # Parse the data
    df = preprocess_input(data_str)
    
    # Rank algorithms
    algo_rank = rank_algorithms(df)
    
    # Calculate average SMAPE
    average_smape = calculate_average_smape(df)
    
    # Generate summary
    summary = generate_summary(algo_rank, average_smape)
    
    # Display the results
    print("Algorithm Ranking Based on Best SMAPE Counts: \n", algo_rank)
    print("Average SMAPE per Algorithm:", average_smape, "")
    print("Summary:\n", summary)
    
    # Define the color palette
    colors = ['#206A92', '#2E2E2E', '#1E5631', '#B22222', '#E3C800', '#2E2E2E']
    colors = ['#2E2E2E', '#A9A9A9', '#D3D3D3']
    # Generate plots
    generate_plots(algo_rank, average_smape, df, colors)

if __name__ == "__main__":
    
    #56-Days
    data_str = """
Well	Method	Tipo	R²	SMAPE	MAE
0 	15/9-F-11 	ARIMA 	0.9937 	2.10% 	104316.5656
1 	15/9-F-11 	AutoARIMA 	0.7916 	12.83% 	704442.4648
2 	15/9-F-11 	LinearRegression 	0.9867 	2.87% 	144934.1766
3 	15/9-F-11 	N-Beats 	0.9929 	2.78% 	125651.6880
4 	15/9-F-11 	NHiTS 	0.9780 	4.54% 	222211.6759
5 	15/9-F-11 	NLinear 	0.9612 	6.15% 	305434.5649
6 	15/9-F-11 	TiDE 	0.9409 	7.50% 	380356.9675
7 	15/9-F-11 	TiDE+RIN 	0.9618 	6.15% 	303646.4913
8 	15/9-F-12 	ARIMA 	0.9817 	3.52% 	805106.0839
9 	15/9-F-12 	AutoARIMA 	0.9813 	3.55% 	814065.3877
10 	15/9-F-12 	LinearRegression 	-17.4622 	49.32% 	19913968.5934
11 	15/9-F-12 	N-Beats 	-0.6439 	26.41% 	7575499.1302
12 	15/9-F-12 	NHiTS 	-0.1547 	22.66% 	6347661.5475
13 	15/9-F-12 	NLinear 	0.9738 	4.00% 	950840.8623
14 	15/9-F-12 	TiDE 	-0.4400 	24.81% 	7078373.8372
15 	15/9-F-12 	TiDE+RIN 	0.1446 	19.75% 	5453737.6322
16 	15/9-F-14 	ARIMA 	0.9651 	4.49% 	955736.9370
17 	15/9-F-14 	AutoARIMA 	0.9667 	4.39% 	933401.6737
18 	15/9-F-14 	LinearRegression 	-27.6223 	61.11% 	23602947.1429
19 	15/9-F-14 	N-Beats 	0.0704 	22.23% 	5068375.9068
20 	15/9-F-14 	NHiTS 	0.6871 	13.63% 	2946205.2597
21 	15/9-F-14 	NLinear 	0.7884 	11.23% 	2411960.0538
22 	15/9-F-14 	TiDE 	0.6320 	14.64% 	3190397.5592
23 	15/9-F-14 	TiDE+RIN 	0.6178 	14.88% 	3250946.0351
24 	15/9-F-15  	ARIMA 	0.9912 	2.37% 	14298.3179
25 	15/9-F-15  	AutoARIMA 	0.9913 	2.36% 	14214.4423
26 	15/9-F-15  	LinearRegression 	0.1698 	17.93% 	140137.9373
27 	15/9-F-15  	N-Beats 	0.9371 	4.84% 	34667.4485
28 	15/9-F-15  	NHiTS 	0.8696 	7.65% 	52591.4533
29 	15/9-F-15  	NLinear 	0.9946 	1.81% 	11656.8908
30 	15/9-F-15  	TiDE 	0.8276 	8.96% 	60935.3897
31 	15/9-F-15  	TiDE+RIN 	0.9888 	2.12% 	14984.7458
32 	Prod-1 	ARIMA 	0.9995 	0.80% 	29820.6990
33 	Prod-1 	AutoARIMA 	0.9995 	0.79% 	29140.0004
34 	Prod-1 	LinearRegression 	0.9154 	7.40% 	335600.4665
35 	Prod-1 	N-Beats 	0.9998 	0.51% 	17300.7420
36 	Prod-1 	NHiTS 	0.9984 	1.56% 	54628.0587
37 	Prod-1 	NLinear 	0.9999 	0.31% 	10580.0574
38 	Prod-1 	TiDE 	1.0000 	0.23% 	7593.1359
39 	Prod-1 	TiDE+RIN 	1.0000 	0.27% 	9172.6162
40 	Prod-10 	ARIMA 	1.0000 	0.25% 	14640.5427
41 	Prod-10 	AutoARIMA 	0.9907 	4.02% 	237714.1962
42 	Prod-10 	LinearRegression 	0.9631 	6.23% 	433203.0204
43 	Prod-10 	N-Beats 	0.9868 	4.88% 	284788.4827
44 	Prod-10 	NHiTS 	0.9958 	2.72% 	160787.0477
45 	Prod-10 	NLinear 	1.0000 	0.25% 	14792.8912
46 	Prod-10 	TiDE 	0.9895 	4.34% 	254171.4145
47 	Prod-10 	TiDE+RIN 	0.9902 	4.12% 	243370.4112
48 	Prod-2 	ARIMA 	0.9987 	0.77% 	37177.6332
49 	Prod-2 	AutoARIMA 	0.9184 	4.57% 	248117.4918
50 	Prod-2 	LinearRegression 	0.9166 	4.69% 	251875.2431
51 	Prod-2 	N-Beats 	0.9988 	1.56% 	49759.2883
52 	Prod-2 	NHiTS 	0.9968 	1.71% 	71428.2615
53 	Prod-2 	NLinear 	0.9994 	0.53% 	25446.5311
54 	Prod-2 	TiDE 	0.9997 	0.52% 	21082.9696
55 	Prod-2 	TiDE+RIN 	0.9996 	0.52% 	21742.8702
56 	Prod-3 	ARIMA 	1.0000 	0.06% 	2313.2969
57 	Prod-3 	AutoARIMA 	1.0000 	0.07% 	2465.0337
58 	Prod-3 	LinearRegression 	0.9991 	0.92% 	38900.1059
59 	Prod-3 	N-Beats 	0.9970 	2.23% 	83802.2065
60 	Prod-3 	NHiTS 	0.9963 	2.46% 	92413.9178
61 	Prod-3 	NLinear 	1.0000 	0.06% 	2382.8636
62 	Prod-3 	TiDE 	0.9992 	1.14% 	43196.8608
63 	Prod-3 	TiDE+RIN 	0.9997 	0.74% 	28284.9606
64 	Prod-4 	ARIMA 	0.9998 	0.46% 	27110.6070
65 	Prod-4 	AutoARIMA 	0.9591 	8.10% 	420344.5145
66 	Prod-4 	LinearRegression 	0.9125 	7.22% 	501887.9347
67 	Prod-4 	N-Beats 	0.9769 	6.29% 	321701.3519
68 	Prod-4 	NHiTS 	0.9931 	3.42% 	176663.1850
69 	Prod-4 	NLinear 	0.9998 	0.45% 	26355.0984
70 	Prod-4 	TiDE 	0.9767 	6.32% 	323528.0813
71 	Prod-4 	TiDE+RIN 	0.9866 	4.69% 	243551.6004
72 	Prod-5 	ARIMA 	1.0000 	0.21% 	7832.3522
73 	Prod-5 	AutoARIMA 	0.9395 	10.67% 	574016.9559
74 	Prod-5 	LinearRegression 	0.9903 	3.70% 	226024.3695
75 	Prod-5 	N-Beats 	0.9209 	12.40% 	658181.4165
76 	Prod-5 	NHiTS 	0.9674 	7.73% 	421558.9011
77 	Prod-5 	NLinear 	1.0000 	0.12% 	4549.4118
78 	Prod-5 	TiDE 	0.9532 	9.37% 	505705.4264
79 	Prod-5 	TiDE+RIN 	0.9509 	9.60% 	517951.8240
80 	Prod-6 	ARIMA 	0.9999 	0.46% 	20019.9874
81 	Prod-6 	AutoARIMA 	0.9966 	2.29% 	102452.8282
82 	Prod-6 	LinearRegression 	0.9764 	4.03% 	223935.7028
83 	Prod-6 	N-Beats 	0.9632 	8.02% 	340309.7095
84 	Prod-6 	NHiTS 	0.9966 	2.30% 	102274.7269
85 	Prod-6 	NLinear 	0.9999 	0.47% 	21295.9518
86 	Prod-6 	TiDE 	0.9968 	2.30% 	100860.4054
87 	Prod-6 	TiDE+RIN 	0.9978 	1.86% 	83249.3334
88 	Prod-7 	ARIMA 	0.9996 	0.59% 	30309.5095
89 	Prod-7 	AutoARIMA 	0.9985 	1.51% 	68220.3472
90 	Prod-7 	LinearRegression 	0.9380 	5.77% 	348151.0450
91 	Prod-7 	N-Beats 	0.9867 	4.65% 	205246.0540
92 	Prod-7 	NHiTS 	0.9841 	5.08% 	224192.0478
93 	Prod-7 	NLinear 	0.9997 	0.50% 	25881.9019
94 	Prod-7 	TiDE 	0.9953 	2.78% 	123044.2631
95 	Prod-7 	TiDE+RIN 	0.9961 	2.48% 	111480.1544
96 	Prod-8 	ARIMA 	1.0000 	0.26% 	9945.6289
97 	Prod-8 	AutoARIMA 	0.9396 	10.61% 	431768.2511
98 	Prod-8 	LinearRegression 	0.9795 	5.03% 	237182.4542
99 	Prod-8 	N-Beats 	0.9920 	3.74% 	157661.1615
100 	Prod-8 	NHiTS 	0.9494 	9.75% 	396480.8124
101 	Prod-8 	NLinear 	0.9971 	2.21% 	94837.8821
102 	Prod-8 	TiDE 	0.9790 	6.16% 	255356.4556
103 	Prod-8 	TiDE+RIN 	0.9735 	6.90% 	286305.4247
104 	Prod-9 	ARIMA 	0.9999 	0.32% 	13177.8944
105 	Prod-9 	AutoARIMA 	0.9501 	9.50% 	438768.2377
106 	Prod-9 	LinearRegression 	0.9709 	5.88% 	314216.2337
107 	Prod-9 	N-Beats 	0.9341 	11.14% 	506406.7651
108 	Prod-9 	NHiTS 	0.9330 	11.23% 	510590.1652
109 	Prod-9 	NLinear 	0.9998 	0.51% 	25986.8877
110 	Prod-9 	TiDE 	0.9853 	5.08% 	239173.8407
111 	Prod-9 	TiDE+RIN 	0.9800 	5.92% 	278575.0576
112 	Load 	ARIMA 	1.0000 	0.32% 	222223.8354
113 	Load 	AutoARIMA 	0.8072 	15.22% 	15917045.5979
114 	Load 	LinearRegression 	0.9250 	9.69% 	9876563.3821
115 	Load 	N-Beats 	0.9986 	1.63% 	1439435.3091
116 	Load 	NHiTS 	0.9834 	5.50% 	4891287.1352
117 	Load 	NLinear 	1.0000 	0.30% 	203301.7143
118 	Load 	TiDE 	0.9995 	0.97% 	863696.5788
119 	Load 	TiDE+RIN 	0.9783 	5.83% 	5413891.6391
120 	Solar 	ARIMA 	0.9989 	3.04% 	41356.7385
121 	Solar 	AutoARIMA 	-12.2415 	72.37% 	3895626.8192
122 	Solar 	LinearRegression 	-0.0786 	40.17% 	1260132.3981
123 	Solar 	N-Beats 	-16.3097 	99.38% 	5028170.8308
124 	Solar 	NHiTS 	-37.5443 	119.02% 	7505793.7667
125 	Solar 	NLinear 	-1.7337 	56.74% 	2000302.3446
126 	Solar 	TiDE 	-7.0529 	80.77% 	3432809.3985
127 	Solar 	TiDE+RIN 	-6.1675 	77.98% 	3237971.3377
128 	Wind 	ARIMA 	0.9459 	8.15% 	870471.1148
129 	Wind 	AutoARIMA 	0.9817 	9.86% 	640447.5073
130 	Wind 	LinearRegression 	0.9685 	11.14% 	775771.2428
131 	Wind 	N-Beats 	0.9961 	4.35% 	303638.6379
132 	Wind 	NHiTS 	0.9843 	5.00% 	511273.3299
133 	Wind 	NLinear 	0.9997 	1.15% 	67629.6299
134 	Wind 	TiDE 	0.9992 	2.40% 	128613.4473
135 	Wind 	TiDE+RIN 	0.9892 	5.22% 	465587.5224
136	15/9-F-11	Ours	NA	NA	2.41	15790.54
137	15/9-F-12	Ours	NA	NA	0.76	16364.07
138	15/9-F-14	Ours	NA	NA	0.68	13797.27
139	15/9-F-15 	Ours	NA	NA	2.33	2269.23
140	Prod-1	Ours	NA	NA	0.27	4707.62
141	Prod-10	Ours	NA	NA	0.19	4851.56
142	Prod-2	Ours	NA	NA	0.25	4499.12
143	Prod-3	Ours	NA	NA	0.16	3505.40
144	Prod-4	Ours	NA	NA	0.23	5497.84
145	Prod-5	Ours	NA	NA	0.22	5205.61
146	Prod-6	Ours	NA	NA	0.16	3476.86
147	Prod-7	Ours	NA	NA	0.20	4503.12
148	Prod-8	Ours	NA	NA	0.18	3661.91
149	Prod-9	Ours	NA	NA	0.15	3652.64
150	Load	Ours	NA	NA	0.24	135404.90
151	Solar	Ours	NA	NA	2.34	23510.95
152	Wind	Ours	NA	NA	0.95	51821.22
155	15/9-F-12	XGB+Ours	0.9987	0.77%	17587.9352
156	15/9-F-14	XGB+Ours	0.9991	0.78%	13751.0892
157	15/9-F-11	XGB+Ours	0.9937	2.99%	18452.9419
158	15/9-F-15   XGB+Ours	0.9875	2.62%	2514.6657
159	Prod-5	XGB+Ours	1.0000	0.27%	4421.9861
160	Prod-6	XGB+Ours	1.0000	0.29%	3595.1340
161	Prod-4	XGB+Ours	1.0000	0.27%	3828.0867
161	Prod-10	XGB+Ours	1.0000	0.28%	4037.7795
163	Prod-8	XGB+Ours	1.0000	0.27%	3226.0021
164	Prod-9	XGB+Ours	1.0000	0.27%	3407.6862
165	Prod-7	XGB+Ours	1.0000	0.32%	4384.3384
166	Prod-2	XGB+Ours	1.0000	0.39%	5152.2744
167	Prod-3	XGB+Ours	1.0000	0.42%	4778.1085
168	Prod-1	XGB+Ours	0.9999	0.46%	5754.6735
169	Wind	XGB+Ours	0.9993	1.52%	2659249
170	Load	XGB+Ours	0.9997	0.74%	13190659
171	Solar	XGB+Ours	0.9960	3.13%	1662282
    """
    
# 112-Days
    data_str = """
Well	Method	Tipo	R²	SMAPE	MAE
0 	15/9-F-11 	ARIMA 	0.9711 	3.92% 	203721.5005
1 	15/9-F-11 	AutoARIMA 	0.9692 	4.06% 	213316.2716
2 	15/9-F-11 	LinearRegression 	0.9760 	3.68% 	195691.4057
3 	15/9-F-11 	N-Beats 	0.9929 	2.78% 	125651.6880
4 	15/9-F-11 	NHiTS 	0.9785 	3.64% 	179920.3210
5 	15/9-F-11 	NLinear 	0.9465 	5.05% 	280336.2954
6 	15/9-F-11 	TiDE 	0.9798 	3.58% 	177589.3793
7 	15/9-F-11 	TiDE+RIN 	0.9527 	4.85% 	264960.2161
8 	15/9-F-12 	ARIMA 	0.6899 	9.37% 	2539289.2222
9 	15/9-F-12 	AutoARIMA 	0.7048 	9.23% 	2487897.0856
10 	15/9-F-12 	LinearRegression 	-15.9788 	39.25% 	15334126.8248
11 	15/9-F-12 	N-Beats 	-0.6439 	26.41% 	7575499.1302
12 	15/9-F-12 	NHiTS 	-1.0279 	22.84% 	6614197.2931
13 	15/9-F-12 	NLinear 	0.0973 	15.85% 	4405144.2872
14 	15/9-F-12 	TiDE 	0.0377 	16.31% 	4547786.5466
15 	15/9-F-12 	TiDE+RIN 	-0.0033 	16.63% 	4645482.0715
16 	15/9-F-14 	ARIMA 	0.8984 	5.88% 	1332275.2258
17 	15/9-F-14 	AutoARIMA 	0.8935 	6.02% 	1364536.0776
18 	15/9-F-14 	LinearRegression 	-9.0025 	37.44% 	11708553.4521
19 	15/9-F-14 	N-Beats 	0.0704 	22.23% 	5068375.9068
20 	15/9-F-14 	NHiTS 	0.9654 	3.29% 	744892.8361
21 	15/9-F-14 	NLinear 	0.9183 	5.27% 	1189928.4424
22 	15/9-F-14 	TiDE 	0.9795 	2.64% 	585620.9327
23 	15/9-F-14 	TiDE+RIN 	0.9653 	3.44% 	765653.3152
24 	15/9-F-15  	ARIMA 	0.9820 	2.19% 	14624.1501
25 	15/9-F-15  	AutoARIMA 	-2.2985 	31.67% 	205260.4168
26 	15/9-F-15  	LinearRegression 	0.8459 	6.34% 	48510.8785
27 	15/9-F-15  	N-Beats 	0.9371 	4.84% 	34667.4485
28 	15/9-F-15  	NHiTS 	0.9572 	3.28% 	24233.1140
29 	15/9-F-15  	NLinear 	0.9821 	2.18% 	14554.1420
30 	15/9-F-15  	TiDE 	0.8278 	5.69% 	43741.2457
31 	15/9-F-15  	TiDE+RIN 	0.9660 	3.02% 	22019.1333
32 	Prod-1 	ARIMA 	0.9978 	1.58% 	59317.6535
33 	Prod-1 	AutoARIMA 	0.9979 	1.55% 	58220.7082
34 	Prod-1 	LinearRegression 	0.8902 	8.12% 	367571.7808
35 	Prod-1 	N-Beats 	0.9998 	0.51% 	17300.7420
36 	Prod-1 	NHiTS 	0.9999 	0.32% 	10822.2511
37 	Prod-1 	NLinear 	0.9987 	1.21% 	44954.1760
38 	Prod-1 	TiDE 	0.9997 	0.61% 	21722.5259
39 	Prod-1 	TiDE+RIN 	0.9986 	1.27% 	47314.1175
40 	Prod-10 	ARIMA 	0.9998 	0.38% 	24631.7797
41 	Prod-10 	AutoARIMA 	0.9883 	4.13% 	256186.7144
42 	Prod-10 	LinearRegression 	0.9811 	4.05% 	291238.5883
43 	Prod-10 	N-Beats 	0.9868 	4.88% 	284788.4827
44 	Prod-10 	NHiTS 	0.9799 	5.59% 	338683.3075
45 	Prod-10 	NLinear 	0.9986 	1.42% 	89400.1051
46 	Prod-10 	TiDE 	0.9856 	4.65% 	285594.3139
47 	Prod-10 	TiDE+RIN 	0.9966 	2.23% 	139355.9911
48 	Prod-2 	ARIMA 	0.9952 	1.34% 	66517.2514
49 	Prod-2 	AutoARIMA 	0.9991 	1.04% 	38118.1282
50 	Prod-2 	LinearRegression 	0.9682 	3.00% 	153373.6952
51 	Prod-2 	N-Beats 	0.9988 	1.56% 	49759.2883
52 	Prod-2 	NHiTS 	0.9972 	1.03% 	51000.1865
53 	Prod-2 	NLinear 	0.9979 	0.95% 	45000.3219
54 	Prod-2 	TiDE 	0.9980 	1.82% 	58793.3390
55 	Prod-2 	TiDE+RIN 	0.9982 	0.94% 	43006.2158
56 	Prod-3 	ARIMA 	1.0000 	0.10% 	4039.6958
57 	Prod-3 	AutoARIMA 	0.9997 	0.46% 	21484.4091
58 	Prod-3 	LinearRegression 	0.9995 	0.59% 	27529.0974
59 	Prod-3 	N-Beats 	0.9970 	2.23% 	83802.2065
60 	Prod-3 	NHiTS 	1.0000 	0.06% 	1924.4186
61 	Prod-3 	NLinear 	1.0000 	0.16% 	6975.3844
62 	Prod-3 	TiDE 	0.9986 	1.33% 	53896.4335
63 	Prod-3 	TiDE+RIN 	1.0000 	0.07% 	2674.2721
64 	Prod-4 	ARIMA 	0.9992 	0.78% 	49176.8227
65 	Prod-4 	AutoARIMA 	0.9555 	7.77% 	420337.1640
66 	Prod-4 	LinearRegression 	0.8980 	7.23% 	515335.1743
67 	Prod-4 	N-Beats 	0.9769 	6.29% 	321701.3519
68 	Prod-4 	NHiTS 	0.9591 	7.70% 	409424.9788
69 	Prod-4 	NLinear 	0.9997 	0.69% 	34051.3861
70 	Prod-4 	TiDE 	0.9951 	2.71% 	144216.5754
71 	Prod-4 	TiDE+RIN 	0.9888 	3.97% 	214900.9484
72 	Prod-5 	ARIMA 	1.0000 	0.24% 	9876.4633
73 	Prod-5 	AutoARIMA 	0.9212 	11.17% 	629703.8136
74 	Prod-5 	LinearRegression 	0.9863 	4.19% 	261474.5798
75 	Prod-5 	N-Beats 	0.9209 	12.40% 	658181.4165
76 	Prod-5 	NHiTS 	0.9807 	5.33% 	310694.4683
77 	Prod-5 	NLinear 	0.9994 	0.82% 	52624.2524
78 	Prod-5 	TiDE 	0.9856 	4.56% 	267880.2927
79 	Prod-5 	TiDE+RIN 	0.9743 	6.17% 	358495.8169
80 	Prod-6 	ARIMA 	0.9996 	0.66% 	32479.1464
81 	Prod-6 	AutoARIMA 	0.9945 	2.73% 	125475.4327
82 	Prod-6 	LinearRegression 	0.9416 	6.09% 	344076.5182
83 	Prod-6 	N-Beats 	0.9632 	8.02% 	340309.7095
84 	Prod-6 	NHiTS 	0.9971 	2.05% 	92520.4520
85 	Prod-6 	NLinear 	0.9978 	1.75% 	79916.2658
86 	Prod-6 	TiDE 	1.0000 	0.22% 	7031.3347
87 	Prod-6 	TiDE+RIN 	0.9978 	1.76% 	80781.9351
88 	Prod-7 	ARIMA 	0.9987 	0.94% 	53151.9599
89 	Prod-7 	AutoARIMA 	0.9929 	3.04% 	141644.6071
90 	Prod-7 	LinearRegression 	0.8977 	6.76% 	422350.3730
91 	Prod-7 	N-Beats 	0.9867 	4.65% 	205246.0540
92 	Prod-7 	NHiTS 	0.9856 	4.49% 	204833.7714
93 	Prod-7 	NLinear 	0.9971 	2.06% 	93425.0477
94 	Prod-7 	TiDE 	0.9951 	2.68% 	121096.7941
95 	Prod-7 	TiDE+RIN 	0.9959 	2.40% 	109775.3659
96 	Prod-8 	ARIMA 	0.9998 	0.37% 	15727.6835
97 	Prod-8 	AutoARIMA 	0.9191 	11.29% 	480590.2376
98 	Prod-8 	LinearRegression 	0.9853 	3.95% 	191535.0003
99 	Prod-8 	N-Beats 	0.9920 	3.74% 	157661.1615
100 	Prod-8 	NHiTS 	0.9817 	5.26% 	229840.8098
101 	Prod-8 	NLinear 	0.9999 	0.34% 	15578.2467
102 	Prod-8 	TiDE 	0.9784 	5.70% 	249305.8831
103 	Prod-8 	TiDE+RIN 	0.9826 	5.08% 	223514.1080
104 	Prod-9 	ARIMA 	0.9998 	0.43% 	20410.1269
105 	Prod-9 	AutoARIMA 	0.9286 	10.47% 	504533.6197
106 	Prod-9 	LinearRegression 	0.9824 	4.18% 	230412.9447
107 	Prod-9 	N-Beats 	0.9341 	11.14% 	506406.7651
108 	Prod-9 	NHiTS 	0.9952 	2.63% 	131971.4992
109 	Prod-9 	NLinear 	0.9988 	1.27% 	65289.2235
110 	Prod-9 	TiDE 	0.9775 	5.78% 	284574.0692
111 	Prod-9 	TiDE+RIN 	0.9833 	4.94% 	244823.7438
112 	Load 	ARIMA 	0.9999 	0.50% 	365326.8422
113 	Load 	AutoARIMA 	0.9431 	9.40% 	8611442.4926
114 	Load 	LinearRegression 	0.9929 	2.24% 	2553685.6055
115 	Load 	N-Beats 	0.9986 	1.63% 	1439435.3091
116 	Load 	NHiTS 	0.9298 	10.65% 	9634067.4447
117 	Load 	NLinear 	0.9583 	7.96% 	7352193.6550
118 	Load 	TiDE 	0.9388 	9.91% 	8991595.2096
119 	Load 	TiDE+RIN 	0.7942 	18.13% 	16172141.7404
120 	Solar 	ARIMA 	0.9965 	4.19% 	68257.4438
121 	Solar 	AutoARIMA 	-1.0899 	50.65% 	1738019.9021
122 	Solar 	LinearRegression 	-59.3890 	124.99% 	9139337.8324
123 	Solar 	N-Beats 	-16.3097 	99.38% 	5028170.8308
124 	Solar 	NHiTS 	-140.6040 	142.67% 	13931051.5394
125 	Solar 	NLinear 	0.6458 	24.69% 	709592.3444
126 	Solar 	TiDE 	-148.8999 	144.07% 	14364875.1503
127 	Solar 	TiDE+RIN 	-123.2438 	140.33% 	13077996.4100
128 	Wind 	ARIMA 	0.9964 	3.08% 	241328.1621
129 	Wind 	AutoARIMA 	0.9978 	2.46% 	188255.0598
130 	Wind 	LinearRegression 	0.5333 	32.42% 	2893990.6143
131 	Wind 	N-Beats 	0.9961 	4.35% 	303638.6379
132 	Wind 	NHiTS 	0.6720 	26.62% 	2437999.7758
133 	Wind 	NLinear 	0.5939 	32.54% 	2781021.9545
134 	Wind 	TiDE 	0.7286 	24.19% 	2227066.4279
135 	Wind 	TiDE+RIN 	0.5099 	36.40% 	3055949.1540
136	15/9-F-11	Ours	NA	NA	7.26%	62,977.05
137	15/9-F-12	Ours	NA	NA	1.63%	52,892.49
138	15/9-F-14	Ours	NA	NA	1.26%	32,491.13
139	15/9-F-15 	Ours	NA	NA	4.20%	5,625.87
140	Prod-1	Ours	NA	NA	0.37%	10,269.41
141	Prod-10	Ours	NA	NA	0.45%	14,704.24
142	Prod-2	Ours	NA	NA	0.37%	12,066.81
143	Prod-3	Ours	NA	NA	0.56%	13,483.83
144	Prod-4	Ours	NA	NA	0.44%	12,794.17
145	Prod-5	Ours	NA	NA	0.31%	9,972.45
146	Prod-6	Ours	NA	NA	0.32%	9,091.69
147	Prod-7	Ours	NA	NA	0.50%	14,313.49
148	Prod-8	Ours	NA	NA	0.39%	9,643.38
149	Prod-9	Ours	NA	NA	0.40%	10,495.53
150	Load	Ours	NA	NA	0.73%	439,400.29
151	Solar	Ours	NA	NA	4.52%	75,134.08
152	Wind	Ours	NA	NA	2.09%	133,172.01
153	Prod-5	XGB+Ours		0.9999	0.62%	15523.0964
154	Prod-6	XGB+Ours		0.9999	0.66%	14198.8021
155	Prod-4	XGB+Ours		0.9999	0.68%	18707.7931
156	Prod-10	XGB+Ours		0.9999	0.67%	20102.6995
157	Prod-8	XGB+Ours		0.9999	0.66%	14261.4650
158	Prod-9	XGB+Ours		0.9999	0.66%	15829.5929
159	Prod-7	XGB+Ours		0.9998	0.78%	20100.3241
160	Prod-2	XGB+Ours		0.9995	1.01%	24133.9638
161	Prod-3	XGB+Ours		0.9997	0.99%	21350.7224
162	Prod-1	XGB+Ours		0.9994	1.07%	24348.1458
163	15/9-F-12	XGB+Ours		0.9727	1.62%	55138.6914
164	15/9-F-14	XGB+Ours		0.9920	1.30%	33764.3159
165	15/9-F-11	XGB+Ours		0.8660	7.38%	64076.3675
166	15/9-F-15	XGB+Ours		0.7262	4.18%	5616.3411
167	Wind	XGB+Ours		0.9970	2.61%	193502.1969
168	Load	XGB+Ours		0.9991	1.04%	812262.0120
169	Solar	XGB+Ours		0.9898	4.67%	87070.1049
"""
    main(data_str)
    
    table_smape = generate_latex_tables(data_str, best_caption_summary="The approach was the best in 12 out of 17 datasets.")
    print(table_smape)      # Para a tabela apenas com SMAPE